In [4]:
import asyncio, sqlite3
import numpy as np
from dotenv import load_dotenv
from dotvectordb.plugins.embedding_providers.openai_embeddings import OpenAIEmbeddingModel

load_dotenv()
DATABASE_PATH = "TEST_TAGS22.db"
ENTITY_NAME = "shrek"  # Change this to any entity in the database

conn = sqlite3.connect(DATABASE_PATH)
entity = conn.execute(
    "SELECT entity_id, entity_name, entity_description FROM entity WHERE lower(entity_name) = lower(?)",
    (ENTITY_NAME,),
).fetchone()
if entity is None:
    available = [row[0] for row in conn.execute("SELECT entity_name FROM entity ORDER BY entity_name")]
    conn.close()
    raise ValueError(f"Entity {ENTITY_NAME!r} was not found. Available entities: {available}")

entity_id, entity_name, entity_description = entity
current_groups = [row[0] for row in conn.execute(
    "SELECT entity_type FROM entity_to_entity_type_mapping WHERE entity_id = ? ORDER BY entity_type",
    (entity_id,),
)]
entity_types = conn.execute(
    "SELECT entity_type, type_description FROM entity_types ORDER BY entity_type"
).fetchall()
conn.close()

embedding_model = OpenAIEmbeddingModel()
entity_text = f"Entity name: {entity_name}\nEntity description: {entity_description}"
type_texts = [f"Entity group: {name}\nDefinition: {description}" for name, description in entity_types]
entity_result, *type_results = await asyncio.gather(
    embedding_model.embed(entity_text),
    *(embedding_model.embed(text) for text in type_texts),
)
entity_embedding = entity_result[0]
type_embeddings = np.vstack([result[0] for result in type_results])
entity_embedding = entity_embedding / np.linalg.norm(entity_embedding)
type_embeddings = type_embeddings / np.linalg.norm(type_embeddings, axis=1, keepdims=True)
similarities = type_embeddings @ entity_embedding
ranking = sorted(
    [(entity_types[index][0], float(score)) for index, score in enumerate(similarities)],
    key=lambda item: item[1],
    reverse=True,
)

print(f"Entity: {entity_name}")
print(f"Current groups: {current_groups}")
print(f"Description: {entity_description}")
print("\nEmbedding similarity to every entity group:")
for rank, (group, similarity) in enumerate(ranking, start=1):
    print(f"{rank:>2}. {group:<20} {similarity:.4f}")


Entity: shrek
Current groups: ['WORK_OF_ART', 'individual', 'organization']
Description: Shrek is an ogre who values his privacy and is initially reluctant to form friendships.<SEP>Shrek is a fictional character who is depicted as a large, green ogre and is known for his adventures and humorous personality.<SEP>Shrek is a character who is referred to as a champion and a knight, known for his unorthodox approach to rescuing Princess Fiona.<SEP>Shrek is a character who expresses his feelings about Princess Fiona and is protective of his swamp.<SEP>Shrek is an ogre who is tasked with rescuing a princess and is characterized by his gruff demeanor and desire for solitude.<SEP>Shrek is a fictional character who is an ogre and expresses feelings of isolation and frustration with how others perceive him.<SEP>Shrek is an ogre who is known for his unique appearance and personality, and he is a central character in the story involving Princess Fiona.<SEP>Shrek is a fictional ogre who desires priv